# 学習済み生徒モデルでロボットを動かして、提出用の動画を作る

`CaP_Distillation_Gemini_Teacher_JP.ipynb` で学習した **生徒モデル（Qwen2.5-Coder + LoRA）** を Google Drive から読み込み、決めておいたタスクを順番に実行して、**1 本の提出用動画（mp4）** にまとめます。学習はしません。

## このノートブックでやること

1. Drive の `cap_distill/student_lora` から LoRA アダプタを読み込む（ベースモデル名はアダプタの設定から自動で読む）
2. `DEMO_TASKS` のタスクを 1 つずつ実行し、シミュレーションを録画する
   - テンプレートのタスクは **成功／失敗を自動判定** して動画に表示する
   - 画面右側に **生徒が生成したコード** を表示する
   - `COMPARE_WITH_TEACHER = True` なら、同じシーンで教師（Gemini）も動かして **左右に並べる**
3. タイトル画面＋各タスクのクリップ＋結果まとめ画面をつないで、`cap_distill/videos_final/` に保存する
4. 実行記録（指示・生成コード・成否・時間）を `cap_distill/report/demo_video_<実験名>.jsonl` / `.csv` に保存する（レポート用）

## 動画の再生速度

シミュレーションは 1/480 秒刻みで、40 ステップごと（= 12 枚/秒）に画像を保存します。これを `12 × PLAYBACK_SPEED` fps で書き出すので、`PLAYBACK_SPEED = 2.0` なら **ちょうど 2 倍速** です。倍速であることはタイトル画面と各クリップの下部に表示されます（YouTube の概要欄にも書いてください。最後のセルで概要欄の文例を出力します）。

## 使い方

1. **GPU ランタイム**（学習と同じ GPU を推奨。7B を学習したなら A100）を選ぶ
2. 最初の設定セルで `OMNI_ACCOUNT`（omnicampus のアカウント名）などを設定する
3. 「ランタイム → すべてのセルを実行」
4. 途中で止まっても、再実行すれば作成済みのクリップはスキップして続きから作ります

In [ ]:
#@title ⚙️ 設定
#@markdown 学習時と同じ実験名（Drive の記録ファイル名に使われます）
EXPERIMENT_NAME = 'exp1'  #@param {type:"string"}
#@markdown 動画のタイトル画面とファイル名に使う omnicampus のアカウント名
OMNI_ACCOUNT = 'omnicampusアカウント名'  #@param {type:"string"}
#@markdown 同じシーンで教師（Gemini）も動かして左右に並べる（Gemini API キーが必要）
COMPARE_WITH_TEACHER = False  #@param {type:"boolean"}
#@markdown 再生速度（シミュレーション時間に対する倍率）。課題の上限は 2 倍速
PLAYBACK_SPEED = 2.0  #@param {type:"number"}
#@markdown 作成済みのクリップも作り直す（OFF なら続きから）
OVERWRITE_CLIPS = False  #@param {type:"boolean"}
#@markdown 学習から除外したテンプレート（学習時の `HOLDOUT_TEMPLATES` と同じにする。動画で「学習外」と表示されます）
HOLDOUT_TEMPLATES_STR = ''  #@param {type:"string"}

import os
HOLDOUT_TEMPLATES = [t.strip() for t in HOLDOUT_TEMPLATES_STR.split(',') if t.strip()]

# --- Gemini（教師との比較で使う。比較しないならキーは無くてよい）---
gemini_api_key = 'YOUR KEY HERE'
try:
    from google.colab import userdata
    gemini_api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_api_key = os.environ.get('GEMINI_API_KEY', gemini_api_key)
MODEL_NAME = 'gemini-3.5-flash-lite'
REQUESTS_PER_MINUTE = 8

# --- 保存先（学習ノートブックと同じ Drive のフォルダ）---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/cap_distill'
except ImportError:
    SAVE_DIR = './cap_distill'

ADAPTER_DIR = f'{SAVE_DIR}/student_lora'
REPORT_DIR = f'{SAVE_DIR}/report'
FINAL_DIR = f'{SAVE_DIR}/videos_final'
CLIP_DIR = f'{FINAL_DIR}/clips_{EXPERIMENT_NAME}' + ('_compare' if COMPARE_WITH_TEACHER else '')
FINAL_VIDEO_PATH = f'{FINAL_DIR}/PAI最終課題_{OMNI_ACCOUNT}' + ('_compare' if COMPARE_WITH_TEACHER else '') + '.mp4'
DEMO_VIDEO_LOG = f'{REPORT_DIR}/demo_video_{EXPERIMENT_NAME}.jsonl'
for d in (REPORT_DIR, FINAL_DIR, CLIP_DIR):
    os.makedirs(d, exist_ok=True)

if not os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'):
    raise FileNotFoundError(f'学習済みの LoRA アダプタが見つかりません: {ADAPTER_DIR}\n'
                            '学習ノートブックの Part B を実行したか、保存先が合っているか確認してください。')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU がありません。「ランタイム → ランタイムのタイプを変更」で GPU を選んでください。')
print('アダプタ:', ADAPTER_DIR)
print('動画の保存先:', FINAL_VIDEO_PATH)

## 1. ライブラリのインストール

シミュレーション用（学習ノートブックと同じ）・生徒モデル用（`transformers` / `peft`）に加えて、動画に日本語を書き込むために日本語フォント（Noto Sans CJK）を入れます。

In [ ]:
!pip install -q google-genai numpy scipy shapely astunparse pygments pybullet imageio imageio-ffmpeg opencv-python gdown pandas

In [ ]:
# Colab に入っている古い torchao (0.10) は新しい peft と非互換なので削除 (LoRA には不要)
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate
# 動画に日本語を書くためのフォント
!apt-get -qq install -y fonts-noto-cjk > /dev/null

In [ ]:
import re
import ast
import copy
import json
import time
import base64
import random
import threading
from time import sleep

import numpy as np
import pandas as pd
import cv2
import imageio
import pybullet
import pybullet_data
from IPython.display import HTML, display

import shapely
import astunparse
from shapely.geometry import *
from shapely.affinity import *

from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import TerminalFormatter

from google import genai
from google.genai import types
from google.genai import errors as genai_errors
import httpx  # google-genai が内部で使う通信ライブラリ（タイムアウト例外の判定用）

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    import matplotlib.pyplot as plt
    def cv2_imshow(img_bgr):
        plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()


# --- JSONL（1 行 1 レコードの JSON）の読み書き ---
def append_jsonl(path, record):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

## 2. 共通セットアップ（学習ノートブックと同じコード）

ここから「タスクの自動生成と成功判定」までは `CaP_Distillation_Gemini_Teacher_JP.ipynb` のセルをそのまま使っています（Gemini 呼び出し・アセット・LMP・シミュレーション環境・ロボット API・プロンプト・`setup_LMP`・テンプレート）。生徒は few-shot プロンプトを使いませんが、LMP の仕組みは教師と共通なので、プロンプトの定義もそのまま読み込みます。

In [ ]:
# 応答が返ってこないまま固まらないよう、1 リクエスト 60 秒でタイムアウトさせる（単位はミリ秒）
client = genai.Client(api_key=gemini_api_key, http_options=types.HttpOptions(timeout=60_000))

CODE_COMPLETION_SYSTEM_INSTRUCTION = """You are a Python code completion engine.
The user message is a Python script made of example pairs: a comment describing a task, followed by the code that does it.
Continue the script by writing ONLY the Python code for the LAST comment.
Rules:
- Output raw Python code only. No markdown, no code fences, no explanations.
- Do not repeat the last comment line.
- Do not write any import statements; everything needed is already imported.
- Use only the functions and variables that appear in the examples.
- Stop after the code for the last comment. Do not write new comments or new tasks."""

RETRYABLE_STATUS_CODES = (429, 500, 502, 503, 504)
MAX_RETRIES = 6
_last_request_time = [0.0]


def _extract_code(text):
    """```python ... ``` で囲まれていれば中身だけを取り出す。"""
    m = re.search(r'```(?:python|py)?[ \t]*\n(.*?)(?:```|$)', text, re.S)
    return m.group(1) if m else text


def _postprocess(text, prompt, stop_tokens):
    """モデルの出力を「続きのコードだけ」に整える（教師・生徒の両方で使う）。"""
    text = _extract_code(text)
    last_prompt_line = prompt.strip().split('\n')[-1].strip()

    out_lines = []
    for line in text.split('\n'):
        stripped = line.strip()
        if not out_lines and (stripped == '' or stripped == last_prompt_line):
            continue
        if any(stripped.startswith(tok) for tok in stop_tokens):
            break
        if stripped.startswith('import ') or (stripped.startswith('from ') and ' import ' in stripped):
            continue
        out_lines.append(line)
    return '\n'.join(out_lines).strip()


def _throttle():
    """REQUESTS_PER_MINUTE を超えないように、前回の呼び出しからの間隔をあける。"""
    wait = 60.0 / REQUESTS_PER_MINUTE - (time.time() - _last_request_time[0])
    if wait > 0:
        sleep(wait)
    _last_request_time[0] = time.time()


# 思考（thinking）トークンも max_output_tokens に含まれるので、その分の余裕を足しておく
THINKING_TOKEN_HEADROOM = 1024


def _thinking_config(model):
    """思考をできるだけ減らす設定（速く・安く・出力が途中で切れないように）。"""
    if model.startswith('gemini-2.5'):
        # 2.5 系は thinking_budget で指定（Flash / Flash-Lite は 0 でオフ、Pro は最小 128）
        return types.ThinkingConfig(thinking_budget=0 if 'flash' in model else 128)
    # 3 系以降は thinking_level で指定（オフにはできないので最小の minimal にする）
    return types.ThinkingConfig(thinking_level='minimal')


def gemini_complete(prompt, stop=(), temperature=0.0, max_tokens=512, model=None,
                    system_instruction=CODE_COMPLETION_SYSTEM_INSTRUCTION):
    model = model or MODEL_NAME
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=max_tokens + THINKING_TOKEN_HEADROOM,
        thinking_config=_thinking_config(model),
    )

    for attempt in range(MAX_RETRIES):
        _throttle()
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=config)
            return _postprocess(response.text or '', prompt, list(stop))
        except genai_errors.APIError as e:
            if e.code not in RETRYABLE_STATUS_CODES or attempt == MAX_RETRIES - 1:
                raise
            wait = min(10 * 2 ** attempt, 60)
            print(f'Gemini API got err {e.code}. Retrying after {wait}s.')
            sleep(wait)
        except (httpx.TimeoutException, httpx.NetworkError) as e:
            if attempt == MAX_RETRIES - 1:
                raise
            print(f'Gemini API timeout / network error ({type(e).__name__}). Retrying after 10s.')
            sleep(10)

In [ ]:
import os
# ランタイム再接続後は cwd が変わっていることがあるので /content に固定
if os.path.isdir('/content'):
    os.chdir('/content')

ASSETS = {
    'ur5e.zip':           ('1Cc_fDSBL6QiDvNT4dpfAEbhbALSVoWcc', 'ur5e/ur5e.urdf'),
    'robotiq_2f_85.zip':  ('1yOMEm-Zp_DL3nItG9RozPeJAmeOldekX', 'robotiq_2f_85/robotiq_2f_85.urdf'),
    'bowl.zip':           ('1GsqNLhEl9dd4Mc3BM0dX3MibOI1FVWNM', 'bowl/bowl.urdf'),
}
for zip_name, (file_id, urdf) in ASSETS.items():
    if os.path.exists(urdf):
        continue
    if not os.path.exists(zip_name) or os.path.getsize(zip_name) < 1000:
        !gdown --fuzzy {file_id} -O {zip_name}
    !unzip -q -o {zip_name}

missing = [urdf for _, urdf in ASSETS.values() if not os.path.exists(urdf)]
if missing:
    raise FileNotFoundError(
        f'アセットが見つかりません: {missing}\n'
        'gdown のダウンロードに失敗した可能性があります（Google Drive のアクセス制限など）。'
        '少し時間をおいてこのセルを再実行してください。')
print('アセット OK:', os.getcwd())

In [ ]:
def merge_dicts(dicts):
    """複数の辞書を1つにまとめる（後ろの辞書が優先）。"""
    return {
        k: v
        for d in dicts
        for k, v in d.items()
    }


def var_exists(name, all_vars):
    """name という名前の変数・関数が all_vars の中に存在するか。"""
    try:
        eval(name, all_vars)
    except:
        exists = False
    else:
        exists = True
    return exists


def exec_safe(code_str, gvars=None, lvars=None):
    """LLMが生成したコードを、最低限の安全チェックをしてから実行する。"""
    banned_phrases = ['import', '__']
    for phrase in banned_phrases:
        assert phrase not in code_str, f'生成コードに禁止語 "{phrase}" が含まれています:\n{code_str}'

    if gvars is None:
        gvars = {}
    if lvars is None:
        lvars = {}
    # 生成コードの中から exec / eval を使えないようにする
    empty_fn = lambda *args, **kwargs: None
    custom_gvars = merge_dicts([
        gvars,
        {'exec': empty_fn, 'eval': empty_fn}
    ])
    exec(code_str, custom_gvars, lvars)

In [ ]:
class FunctionParser(ast.NodeTransformer):

    def __init__(self, fs, f_assigns):
        super().__init__()
        self._fs = fs                # 関数名 -> 呼び出し式 の文字列
        self._f_assigns = f_assigns  # 関数名 -> 代入文 の文字列

    def visit_Call(self, node):
        self.generic_visit(node)  # 入れ子になった呼び出しも先に処理する
        if isinstance(node.func, ast.Name):  # obj.method() ではなく func() の形だけを対象にする
            f_sig = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.func).strip()
            self._fs[f_name] = f_sig
        return node

    def visit_Assign(self, node):
        self.generic_visit(node)
        if isinstance(node.value, ast.Call):  # 右辺が関数呼び出しの代入文
            assign_str = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.value.func).strip()
            self._f_assigns[f_name] = assign_str
        return node

In [ ]:
VERBOSE = True


def _log(msg):
    if VERBOSE:
        print(msg)


class CallRecorder:
    """LMP の入出力を記録する（教師データの元）。"""

    def __init__(self):
        self.calls = []

    def reset(self):
        self.calls = []

    def record(self, **kwargs):
        self.calls.append(kwargs)


RECORDER = CallRecorder()


def gemini_backend(lmp_name, full_prompt, context, query, history, stop, cfg):
    """教師バックエンド：few-shot プロンプト全体を Gemini に渡して続きを書かせる。"""
    return gemini_complete(
        full_prompt,
        stop=stop,
        temperature=cfg['temperature'],
        max_tokens=cfg['max_tokens'],
        model=cfg['model'],
    )

In [ ]:
class LMP:

    def __init__(self, name, cfg, lmp_fgen, fixed_vars, variable_vars, backend):
        self._name = name
        self._cfg = cfg
        self._base_prompt = self._cfg['prompt_text']
        self._stop_tokens = list(self._cfg['stop'])
        self._lmp_fgen = lmp_fgen
        self._fixed_vars = fixed_vars
        self._variable_vars = variable_vars
        self._backend = backend
        self.exec_hist = ''

    def clear_exec_hist(self):
        self.exec_hist = ''

    def build_prompt(self, query, context=''):
        if len(self._variable_vars) > 0:
            variable_vars_imports_str = f"from utils import {', '.join(self._variable_vars.keys())}"
        else:
            variable_vars_imports_str = ''
        prompt = self._base_prompt.replace('{variable_vars_imports}', variable_vars_imports_str)

        if self._cfg['maintain_session']:
            prompt += f'\n{self.exec_hist}'

        if context != '':
            prompt += f'\n{context}'

        use_query = f'{self._cfg["query_prefix"]}{query}{self._cfg["query_suffix"]}'
        prompt += f'\n{use_query}'

        return prompt, use_query

    def __call__(self, query, context='', **kwargs):
        prompt, use_query = self.build_prompt(query, context=context)
        history = self.exec_hist if self._cfg['maintain_session'] else ''

        # ① バックエンド（教師 or 生徒）にコードを書かせ、入出力を記録する
        t0 = time.time()
        code_str = self._backend(
            lmp_name=self._name, full_prompt=prompt, context=context,
            query=use_query, history=history, stop=self._stop_tokens, cfg=self._cfg,
        )
        RECORDER.record(lmp=self._name, context=context, query=use_query, history=history,
                        completion=code_str, latency=time.time() - t0)

        if self._cfg['include_context'] and context != '':
            to_exec = f'{context}\n{code_str}'
            to_log = f'{context}\n{use_query}\n{code_str}'
        else:
            to_exec = code_str
            to_log = f'{use_query}\n{to_exec}'
        _log(f'LMP {self._name} exec:\n\n{highlight(to_log, PythonLexer(), TerminalFormatter())}\n')

        # ② 未定義関数の生成
        new_fs = self._lmp_fgen.create_new_fs_from_code(code_str)
        self._variable_vars.update(new_fs)

        # ③ 実行
        gvars = merge_dicts([self._fixed_vars, self._variable_vars])
        lvars = kwargs
        if not self._cfg['debug_mode']:
            exec_safe(to_exec, gvars, lvars)

        self.exec_hist += f'\n{to_exec}'
        if self._cfg['maintain_session']:
            self._variable_vars.update(lvars)

        if self._cfg['has_return']:
            return lvars[self._cfg['return_val_name']]


class LMPFGen:

    def __init__(self, cfg, fixed_vars, variable_vars, backend):
        self._cfg = cfg
        self._stop_tokens = list(self._cfg['stop'])
        self._fixed_vars = fixed_vars
        self._variable_vars = variable_vars
        self._base_prompt = self._cfg['prompt_text']
        self._backend = backend

    def create_f_from_sig(self, f_name, f_sig, other_vars=None, fix_bugs=False, return_src=False):
        _log(f'Creating function: {f_sig}')

        use_query = f'{self._cfg["query_prefix"]}{f_sig}{self._cfg["query_suffix"]}'
        prompt = f'{self._base_prompt}\n{use_query}'

        t0 = time.time()
        f_src = self._backend(
            lmp_name='fgen', full_prompt=prompt, context='',
            query=use_query, history='', stop=self._stop_tokens, cfg=self._cfg,
        )
        RECORDER.record(lmp='fgen', context='', query=use_query, history='',
                        completion=f_src, latency=time.time() - t0)

        if other_vars is None:
            other_vars = {}
        gvars = merge_dicts([self._fixed_vars, self._variable_vars, other_vars])
        lvars = {}
        exec_safe(f_src, gvars, lvars)
        f = lvars[f_name]

        to_print = highlight(f'{use_query}\n{f_src}', PythonLexer(), TerminalFormatter())
        _log(f'LMP FGEN created:\n\n{to_print}\n')

        if return_src:
            return f, f_src
        return f

    def create_new_fs_from_code(self, code_str, other_vars=None, fix_bugs=False, return_src=False):
        fs, f_assigns = {}, {}
        f_parser = FunctionParser(fs, f_assigns)
        f_parser.visit(ast.parse(code_str))
        for f_name, f_assign in f_assigns.items():
            if f_name in fs:
                fs[f_name] = f_assign

        if other_vars is None:
            other_vars = {}

        new_fs = {}
        srcs = {}
        for f_name, f_sig in fs.items():
            all_vars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
            if not var_exists(f_name, all_vars):
                f, f_src = self.create_f_from_sig(f_name, f_sig, new_fs, fix_bugs=fix_bugs, return_src=True)

                f_def_body = astunparse.unparse(ast.parse(f_src).body[0].body)
                child_fs, child_f_srcs = self.create_new_fs_from_code(
                    f_def_body, other_vars=all_vars, fix_bugs=fix_bugs, return_src=True
                )

                if len(child_fs) > 0:
                    new_fs.update(child_fs)
                    srcs.update(child_f_srcs)
                    gvars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
                    lvars = {}
                    exec_safe(f_src, gvars, lvars)
                    f = lvars[f_name]

                new_fs[f_name], srcs[f_name] = f, f_src

        if return_src:
            return new_fs, srcs
        return new_fs

In [ ]:
# # Global constants: pick and place objects, colors, workspace bounds
COLORS = {
    'blue':   (78/255,  121/255, 167/255, 255/255),
    'red':    (255/255,  87/255,  89/255, 255/255),
    'green':  (89/255,  169/255,  79/255, 255/255),
    'orange': (242/255, 142/255,  43/255, 255/255),
    'yellow': (237/255, 201/255,  72/255, 255/255),
    'purple': (176/255, 122/255, 161/255, 255/255),
    'pink':   (255/255, 157/255, 167/255, 255/255),
    'cyan':   (118/255, 183/255, 178/255, 255/255),
    'brown':  (156/255, 117/255,  95/255, 255/255),
    'gray':   (186/255, 176/255, 172/255, 255/255),
}

CORNER_POS = {
  'top left corner':     (-0.3 + 0.05, -0.2 - 0.05, 0),
  'top side':            (0,           -0.2 - 0.05, 0),
  'top right corner':    (0.3 - 0.05,  -0.2 - 0.05, 0),
  'left side':           (-0.3 + 0.05, -0.5,        0),
  'middle':              (0,           -0.5,        0),
  'right side':          (0.3 - 0.05,  -0.5,        0),
  'bottom left corner':  (-0.3 + 0.05, -0.8 + 0.05, 0),
  'bottom side':         (0,           -0.8 + 0.05, 0),
  'bottom right corner': (0.3 - 0.05,  -0.8 + 0.05, 0),
}

ALL_BLOCKS = ['blue block', 'red block', 'green block', 'orange block', 'yellow block', 'purple block', 'pink block', 'cyan block', 'brown block', 'gray block']
ALL_BOWLS = ['blue bowl', 'red bowl', 'green bowl', 'orange bowl', 'yellow bowl', 'purple bowl', 'pink bowl', 'cyan bowl', 'brown bowl', 'gray bowl']

PIXEL_SIZE = 0.00267857
BOUNDS = np.float32([[-0.3, 0.3], [-0.8, -0.2], [0, 0.15]])  # X Y Z

In [ ]:
# Gripper (Robotiq 2F85) code

class Robotiq2F85:
  """Gripper handling for Robotiq 2F85."""

  def __init__(self, robot, tool):
    self.robot = robot
    self.tool = tool
    pos = [0.1339999999999999, -0.49199999999872496, 0.5]
    rot = pybullet.getQuaternionFromEuler([np.pi, 0, np.pi])
    urdf = 'robotiq_2f_85/robotiq_2f_85.urdf'
    self.body = pybullet.loadURDF(urdf, pos, rot)
    self.n_joints = pybullet.getNumJoints(self.body)
    self.activated = False

    # Connect gripper base to robot tool.
    pybullet.createConstraint(self.robot, tool, self.body, 0, jointType=pybullet.JOINT_FIXED, jointAxis=[0, 0, 0], parentFramePosition=[0, 0, 0], childFramePosition=[0, 0, -0.07], childFrameOrientation=pybullet.getQuaternionFromEuler([0, 0, np.pi / 2]))

    # Set friction coefficients for gripper fingers.
    for i in range(pybullet.getNumJoints(self.body)):
      pybullet.changeDynamics(self.body, i, lateralFriction=10.0, spinningFriction=1.0, rollingFriction=1.0, frictionAnchor=True)

    # Start thread to handle additional gripper constraints.
    self.motor_joint = 1
    self.constraints_thread = threading.Thread(target=self.step)
    self.constraints_thread.daemon = True
    self.constraints_thread.start()

  # Control joint positions by enforcing hard contraints on gripper behavior.
  # Set one joint as the open/close motor joint (other joints should mimic).
  def step(self):
    while True:
      try:
        currj = [pybullet.getJointState(self.body, i)[0] for i in range(self.n_joints)]
        indj = [6, 3, 8, 5, 10]
        targj = [currj[1], -currj[1], -currj[1], currj[1], currj[1]]
        pybullet.setJointMotorControlArray(self.body, indj, pybullet.POSITION_CONTROL, targj, positionGains=np.ones(5))
      except:
        return
      sleep(0.001)

  # Close gripper fingers.
  def activate(self):
    pybullet.setJointMotorControl2(self.body, self.motor_joint, pybullet.VELOCITY_CONTROL, targetVelocity=1, force=10)
    self.activated = True

  # Open gripper fingers.
  def release(self):
    pybullet.setJointMotorControl2(self.body, self.motor_joint, pybullet.VELOCITY_CONTROL, targetVelocity=-1, force=10)
    self.activated = False

  # If activated and object in gripper: check object contact.
  # If activated and nothing in gripper: check gripper contact.
  # If released: check proximity to surface (disabled).
  def detect_contact(self):
    obj, _, ray_frac = self.check_proximity()
    if self.activated:
      empty = self.grasp_width() < 0.01
      cbody = self.body if empty else obj
      if obj == self.body or obj == 0:
        return False
      return self.external_contact(cbody)
  #   else:
  #     return ray_frac < 0.14 or self.external_contact()

  # Return if body is in contact with something other than gripper
  def external_contact(self, body=None):
    if body is None:
      body = self.body
    pts = pybullet.getContactPoints(bodyA=body)
    pts = [pt for pt in pts if pt[2] != self.body]
    return len(pts) > 0  # pylint: disable=g-explicit-length-test

  def check_grasp(self):
    while self.moving():
      sleep(0.001)
    success = self.grasp_width() > 0.01
    return success

  def grasp_width(self):
    lpad = np.array(pybullet.getLinkState(self.body, 4)[0])
    rpad = np.array(pybullet.getLinkState(self.body, 9)[0])
    dist = np.linalg.norm(lpad - rpad) - 0.047813
    return dist

  def check_proximity(self):
    ee_pos = np.array(pybullet.getLinkState(self.robot, self.tool)[0])
    tool_pos = np.array(pybullet.getLinkState(self.body, 0)[0])
    vec = (tool_pos - ee_pos) / np.linalg.norm((tool_pos - ee_pos))
    ee_targ = ee_pos + vec
    ray_data = pybullet.rayTest(ee_pos, ee_targ)[0]
    obj, link, ray_frac = ray_data[0], ray_data[1], ray_data[2]
    return obj, link, ray_frac

In [ ]:
# Gym-style environment code

class PickPlaceEnv():

  def __init__(self, render=False, high_res=False, high_frame_rate=False):
    self.dt = 1/480
    self.sim_step = 0

    # Configure and start PyBullet.
    # python3 -m pybullet_utils.runServer
    # pybullet.connect(pybullet.SHARED_MEMORY)  # pybullet.GUI for local GUI.
    pybullet.connect(pybullet.DIRECT)  # pybullet.GUI for local GUI.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_GUI, 0)
    pybullet.setPhysicsEngineParameter(enableFileCaching=0)
    assets_path = os.path.abspath("")  # ur5e/ などがある /content
    pybullet.setAdditionalSearchPath(assets_path)
    pybullet.setAdditionalSearchPath(pybullet_data.getDataPath())
    pybullet.setTimeStep(self.dt)

    self.home_joints = (np.pi / 2, -np.pi / 2, np.pi / 2, -np.pi / 2, 3 * np.pi / 2, 0)  # Joint angles: (J0, J1, J2, J3, J4, J5).
    self.home_ee_euler = (np.pi, 0, np.pi)  # (RX, RY, RZ) rotation in Euler angles.
    self.ee_link_id = 9  # Link ID of UR5 end effector.
    self.tip_link_id = 10  # Link ID of gripper finger tips.
    self.gripper = None

    self.render = render
    self.high_res = high_res
    self.high_frame_rate = high_frame_rate

  def reset(self, object_list):
    pybullet.resetSimulation(pybullet.RESET_USE_DEFORMABLE_WORLD)
    pybullet.setGravity(0, 0, -9.8)
    self.cache_video = []

    # Temporarily disable rendering to load URDFs faster.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_RENDERING, 0)

    # Add robot.
    pybullet.loadURDF("plane.urdf", [0, 0, -0.001])
    self.robot_id = pybullet.loadURDF("ur5e/ur5e.urdf", [0, 0, 0], flags=pybullet.URDF_USE_MATERIAL_COLORS_FROM_MTL)
    self.ghost_id = pybullet.loadURDF("ur5e/ur5e.urdf", [0, 0, -10])  # For forward kinematics.
    self.joint_ids = [pybullet.getJointInfo(self.robot_id, i) for i in range(pybullet.getNumJoints(self.robot_id))]
    self.joint_ids = [j[0] for j in self.joint_ids if j[2] == pybullet.JOINT_REVOLUTE]

    # Move robot to home configuration.
    for i in range(len(self.joint_ids)):
      pybullet.resetJointState(self.robot_id, self.joint_ids[i], self.home_joints[i])

    # Add gripper.
    if self.gripper is not None:
      while self.gripper.constraints_thread.is_alive():
        self.constraints_thread_active = False
    self.gripper = Robotiq2F85(self.robot_id, self.ee_link_id)
    self.gripper.release()

    # Add workspace.
    plane_shape = pybullet.createCollisionShape(pybullet.GEOM_BOX, halfExtents=[0.3, 0.3, 0.001])
    plane_visual = pybullet.createVisualShape(pybullet.GEOM_BOX, halfExtents=[0.3, 0.3, 0.001])
    plane_id = pybullet.createMultiBody(0, plane_shape, plane_visual, basePosition=[0, -0.5, 0])
    pybullet.changeVisualShape(plane_id, -1, rgbaColor=[0.2, 0.2, 0.2, 1.0])

    # Load objects according to config.
    self.object_list = object_list
    self.obj_name_to_id = {}
    obj_xyz = np.zeros((0, 3))
    for obj_name in object_list:
      if ('block' in obj_name) or ('bowl' in obj_name):

        # Get random position 15cm+ from other objects.
        while True:
          rand_x = np.random.uniform(BOUNDS[0, 0] + 0.1, BOUNDS[0, 1] - 0.1)
          rand_y = np.random.uniform(BOUNDS[1, 0] + 0.1, BOUNDS[1, 1] - 0.1)
          rand_xyz = np.float32([rand_x, rand_y, 0.03]).reshape(1, 3)
          if len(obj_xyz) == 0:
            obj_xyz = np.concatenate((obj_xyz, rand_xyz), axis=0)
            break
          else:
            nn_dist = np.min(np.linalg.norm(obj_xyz - rand_xyz, axis=1)).squeeze()
            if nn_dist > 0.15:
              obj_xyz = np.concatenate((obj_xyz, rand_xyz), axis=0)
              break
        
        object_color = COLORS[obj_name.split(' ')[0]]
        object_type = obj_name.split(' ')[1]
        object_position = rand_xyz.squeeze()
        if object_type == 'block':
          object_shape = pybullet.createCollisionShape(pybullet.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
          object_visual = pybullet.createVisualShape(pybullet.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
          object_id = pybullet.createMultiBody(0.01, object_shape, object_visual, basePosition=object_position)
        elif object_type == 'bowl':
          object_position[2] = 0
          object_id = pybullet.loadURDF("bowl/bowl.urdf", object_position, useFixedBase=1)
        pybullet.changeVisualShape(object_id, -1, rgbaColor=object_color)
        self.obj_name_to_id[obj_name] = object_id
    

    # Re-enable rendering.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_RENDERING, 1)

    for _ in range(200):
      pybullet.stepSimulation()

    # record object positions at reset
    self.init_pos = {name: self.get_obj_pos(name) for name in object_list}

    return self.get_observation()

  def servoj(self, joints):
    """Move to target joint positions with position control."""
    pybullet.setJointMotorControlArray(
      bodyIndex=self.robot_id,
      jointIndices=self.joint_ids,
      controlMode=pybullet.POSITION_CONTROL,
      targetPositions=joints,
      positionGains=[0.01]*6)
  
  def movep(self, position):
    """Move to target end effector position."""
    joints = pybullet.calculateInverseKinematics(
        bodyUniqueId=self.robot_id,
        endEffectorLinkIndex=self.tip_link_id,
        targetPosition=position,
        targetOrientation=pybullet.getQuaternionFromEuler(self.home_ee_euler),
        maxNumIterations=100)
    self.servoj(joints)

  def get_ee_pos(self):
    ee_xyz = np.float32(pybullet.getLinkState(self.robot_id, self.tip_link_id)[0])
    return ee_xyz

  def step(self, action=None):
    """Do pick and place motion primitive."""
    pick_pos, place_pos = action['pick'].copy(), action['place'].copy()

    # Set fixed primitive z-heights.
    hover_xyz = np.float32([pick_pos[0], pick_pos[1], 0.2])
    if pick_pos.shape[-1] == 2:
      pick_xyz = np.append(pick_pos, 0.025)
    else:
      pick_xyz = pick_pos
      pick_xyz[2] = 0.025
    if place_pos.shape[-1] == 2:
      place_xyz = np.append(place_pos, 0.15)
    else:
      place_xyz = place_pos
      place_xyz[2] = 0.15

    # Move to object.
    ee_xyz = self.get_ee_pos()
    while np.linalg.norm(hover_xyz - ee_xyz) > 0.01:
      self.movep(hover_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    while np.linalg.norm(pick_xyz - ee_xyz) > 0.01:
      self.movep(pick_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    # Pick up object.
    self.gripper.activate()
    for _ in range(240):
      self.step_sim_and_render()
    while np.linalg.norm(hover_xyz - ee_xyz) > 0.01:
      self.movep(hover_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()
    
    for _ in range(50):
      self.step_sim_and_render()
    
    # Move to place location.
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    # Place down object.
    while (not self.gripper.detect_contact()) and (place_xyz[2] > 0.03):
      place_xyz[2] -= 0.001
      self.movep(place_xyz)
      for _ in range(3):
        self.step_sim_and_render()
    self.gripper.release()
    for _ in range(240):
      self.step_sim_and_render()
    place_xyz[2] = 0.2
    ee_xyz = self.get_ee_pos()
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()
    place_xyz = np.float32([0, -0.5, 0.2])
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    observation = self.get_observation()
    reward = self.get_reward()
    done = False
    info = {}
    return observation, reward, done, info

  def set_alpha_transparency(self, alpha: float) -> None:
    for id in range(20):
      visual_shape_data = pybullet.getVisualShapeData(id)
      for i in range(len(visual_shape_data)):
        object_id, link_index, _, _, _, _, _, rgba_color = visual_shape_data[i]
        rgba_color = list(rgba_color[0:3]) +  [alpha]
        pybullet.changeVisualShape(
            self.robot_id, linkIndex=i, rgbaColor=rgba_color)      
        pybullet.changeVisualShape(
            self.gripper.body, linkIndex=i, rgbaColor=rgba_color)

  def step_sim_and_render(self):
    pybullet.stepSimulation()
    self.sim_step += 1

    interval = 40 if self.high_frame_rate else 60
    # Render current image at 8 FPS.
    if self.sim_step % interval == 0 and self.render:
      self.cache_video.append(self.get_camera_image())

  def get_camera_image(self):
    if not self.high_res:
      image_size = (240, 240)
      intrinsics = (120., 0, 120., 0, 120., 120., 0, 0, 1)
    else:
      image_size=(360, 360)
      intrinsics=(180., 0, 180., 0, 180., 180., 0, 0, 1)
    color, _, _, _, _ = env.render_image(image_size, intrinsics)
    return color

  def get_reward(self):
    return None

  def get_observation(self):
    observation = {}

    # Render current image.
    color, depth, position, orientation, intrinsics = self.render_image()

    # Get heightmaps and colormaps.
    points = self.get_pointcloud(depth, intrinsics)
    position = np.float32(position).reshape(3, 1)
    rotation = pybullet.getMatrixFromQuaternion(orientation)
    rotation = np.float32(rotation).reshape(3, 3)
    transform = np.eye(4)
    transform[:3, :] = np.hstack((rotation, position))
    points = self.transform_pointcloud(points, transform)
    heightmap, colormap, xyzmap = self.get_heightmap(points, color, BOUNDS, PIXEL_SIZE)

    observation["image"] = colormap
    observation["xyzmap"] = xyzmap

    return observation

  def render_image(self, image_size=(720, 720), intrinsics=(360., 0, 360., 0, 360., 360., 0, 0, 1)):

    # Camera parameters.
    position = (0, -0.85, 0.4)
    orientation = (np.pi / 4 + np.pi / 48, np.pi, np.pi)
    orientation = pybullet.getQuaternionFromEuler(orientation)
    zrange = (0.01, 10.)
    noise=True

    # OpenGL camera settings.
    lookdir = np.float32([0, 0, 1]).reshape(3, 1)
    updir = np.float32([0, -1, 0]).reshape(3, 1)
    rotation = pybullet.getMatrixFromQuaternion(orientation)
    rotm = np.float32(rotation).reshape(3, 3)
    lookdir = (rotm @ lookdir).reshape(-1)
    updir = (rotm @ updir).reshape(-1)
    lookat = position + lookdir
    focal_len = intrinsics[0]
    znear, zfar = (0.01, 10.)
    viewm = pybullet.computeViewMatrix(position, lookat, updir)
    fovh = (image_size[0] / 2) / focal_len
    fovh = 180 * np.arctan(fovh) * 2 / np.pi

    # Notes: 1) FOV is vertical FOV 2) aspect must be float
    aspect_ratio = image_size[1] / image_size[0]
    projm = pybullet.computeProjectionMatrixFOV(fovh, aspect_ratio, znear, zfar)

    # Render with OpenGL camera settings.
    _, _, color, depth, segm = pybullet.getCameraImage(
        width=image_size[1],
        height=image_size[0],
        viewMatrix=viewm,
        projectionMatrix=projm,
        shadow=1,
        flags=pybullet.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX,
        renderer=pybullet.ER_BULLET_HARDWARE_OPENGL)

    # Get color image.
    color_image_size = (image_size[0], image_size[1], 4)
    color = np.array(color, dtype=np.uint8).reshape(color_image_size)
    color = color[:, :, :3]  # remove alpha channel
    if noise:
      color = np.int32(color)
      color += np.int32(np.random.normal(0, 3, color.shape))
      color = np.uint8(np.clip(color, 0, 255))

    # Get depth image.
    depth_image_size = (image_size[0], image_size[1])
    zbuffer = np.float32(depth).reshape(depth_image_size)
    depth = (zfar + znear - (2 * zbuffer - 1) * (zfar - znear))
    depth = (2 * znear * zfar) / depth
    if noise:
      depth += np.random.normal(0, 0.003, depth.shape)

    intrinsics = np.float32(intrinsics).reshape(3, 3)
    return color, depth, position, orientation, intrinsics

  def get_pointcloud(self, depth, intrinsics):
    """Get 3D pointcloud from perspective depth image.
    Args:
      depth: HxW float array of perspective depth in meters.
      intrinsics: 3x3 float array of camera intrinsics matrix.
    Returns:
      points: HxWx3 float array of 3D points in camera coordinates.
    """
    height, width = depth.shape
    xlin = np.linspace(0, width - 1, width)
    ylin = np.linspace(0, height - 1, height)
    px, py = np.meshgrid(xlin, ylin)
    px = (px - intrinsics[0, 2]) * (depth / intrinsics[0, 0])
    py = (py - intrinsics[1, 2]) * (depth / intrinsics[1, 1])
    points = np.float32([px, py, depth]).transpose(1, 2, 0)
    return points

  def transform_pointcloud(self, points, transform):
    """Apply rigid transformation to 3D pointcloud.
    Args:
      points: HxWx3 float array of 3D points in camera coordinates.
      transform: 4x4 float array representing a rigid transformation matrix.
    Returns:
      points: HxWx3 float array of transformed 3D points.
    """
    padding = ((0, 0), (0, 0), (0, 1))
    homogen_points = np.pad(points.copy(), padding,
                            'constant', constant_values=1)
    for i in range(3):
      points[Ellipsis, i] = np.sum(transform[i, :] * homogen_points, axis=-1)
    return points

  def get_heightmap(self, points, colors, bounds, pixel_size):
    """Get top-down (z-axis) orthographic heightmap image from 3D pointcloud.
    Args:
      points: HxWx3 float array of 3D points in world coordinates.
      colors: HxWx3 uint8 array of values in range 0-255 aligned with points.
      bounds: 3x2 float array of values (rows: X,Y,Z; columns: min,max) defining
        region in 3D space to generate heightmap in world coordinates.
      pixel_size: float defining size of each pixel in meters.
    Returns:
      heightmap: HxW float array of height (from lower z-bound) in meters.
      colormap: HxWx3 uint8 array of backprojected color aligned with heightmap.
      xyzmap: HxWx3 float array of XYZ points in world coordinates.
    """
    width = int(np.round((bounds[0, 1] - bounds[0, 0]) / pixel_size))
    height = int(np.round((bounds[1, 1] - bounds[1, 0]) / pixel_size))
    heightmap = np.zeros((height, width), dtype=np.float32)
    colormap = np.zeros((height, width, colors.shape[-1]), dtype=np.uint8)
    xyzmap = np.zeros((height, width, 3), dtype=np.float32)

    # Filter out 3D points that are outside of the predefined bounds.
    ix = (points[Ellipsis, 0] >= bounds[0, 0]) & (points[Ellipsis, 0] < bounds[0, 1])
    iy = (points[Ellipsis, 1] >= bounds[1, 0]) & (points[Ellipsis, 1] < bounds[1, 1])
    iz = (points[Ellipsis, 2] >= bounds[2, 0]) & (points[Ellipsis, 2] < bounds[2, 1])
    valid = ix & iy & iz
    points = points[valid]
    colors = colors[valid]

    # Sort 3D points by z-value, which works with array assignment to simulate
    # z-buffering for rendering the heightmap image.
    iz = np.argsort(points[:, -1])
    points, colors = points[iz], colors[iz]
    px = np.int32(np.floor((points[:, 0] - bounds[0, 0]) / pixel_size))
    py = np.int32(np.floor((points[:, 1] - bounds[1, 0]) / pixel_size))
    px = np.clip(px, 0, width - 1)
    py = np.clip(py, 0, height - 1)
    heightmap[py, px] = points[:, 2] - bounds[2, 0]
    for c in range(colors.shape[-1]):
      colormap[py, px, c] = colors[:, c]
      xyzmap[py, px, c] = points[:, c]
    colormap = colormap[::-1, :, :]  # Flip up-down.
    xv, yv = np.meshgrid(np.linspace(BOUNDS[0, 0], BOUNDS[0, 1], height),
                         np.linspace(BOUNDS[1, 0], BOUNDS[1, 1], width))
    xyzmap[:, :, 0] = xv
    xyzmap[:, :, 1] = yv
    xyzmap = xyzmap[::-1, :, :]  # Flip up-down.
    heightmap = heightmap[::-1, :]  # Flip up-down.
    return heightmap, colormap, xyzmap
  
  def on_top_of(self, obj_a, obj_b):
    """
    check if obj_a is on top of obj_b
    condition 1: l2 distance on xy plane is less than a threshold
    condition 2: obj_a is higher than obj_b
    """
    obj_a_pos = self.get_obj_pos(obj_a)
    obj_b_pos = self.get_obj_pos(obj_b)
    xy_dist = np.linalg.norm(obj_a_pos[:2] - obj_b_pos[:2])
    if obj_b in CORNER_POS:
      is_near = xy_dist < 0.06
      return is_near
    elif 'bowl' in obj_b:
      is_near = xy_dist < 0.06
      is_higher = obj_a_pos[2] > obj_b_pos[2]
      return is_near and is_higher
    else:
      is_near = xy_dist < 0.04
      is_higher = obj_a_pos[2] > obj_b_pos[2]
      return is_near and is_higher
  
  def get_obj_id(self, obj_name):
    try:
      if obj_name in self.obj_name_to_id:
        obj_id = self.obj_name_to_id[obj_name]
      else:
        obj_name = obj_name.replace('circle', 'bowl').replace('square', 'block').replace('small', '').strip()
        obj_id = self.obj_name_to_id[obj_name]
    except:
      print(f'requested_name="{obj_name}"')
      print(f'available_objects_and_id="{self.obj_name_to_id}')
    return obj_id
  
  def get_obj_pos(self, obj_name):
    obj_name = obj_name.replace('the', '').replace('_', ' ').strip()
    if obj_name in CORNER_POS:
      position = np.float32(np.array(CORNER_POS[obj_name]))
    else:
      pick_id = self.get_obj_id(obj_name)
      pose = pybullet.getBasePositionAndOrientation(pick_id)
      position = np.float32(pose[0])
    return position
  
  def get_bounding_box(self, obj_name):
    obj_id = self.get_obj_id(obj_name)
    return pybullet.getAABB(obj_id)

In [ ]:
class SimTimeout(Exception):
    pass


class SafePickPlaceEnv(PickPlaceEnv):
    MAX_SIM_STEPS_PER_EPISODE = 60000  # dt=1/480 秒なので、シミュレーション時間で約 125 秒

    def reset(self, object_list):
        obs = super().reset(object_list)
        self.episode_start_step = self.sim_step
        return obs

    def step_sim_and_render(self):
        super().step_sim_and_render()
        if self.sim_step - self.episode_start_step > self.MAX_SIM_STEPS_PER_EPISODE:
            raise SimTimeout('シミュレーションのステップ数が上限を超えました')


# 環境は 1 つだけ作って使い回す（get_camera_image がグローバル変数 env を参照するため名前は env）
env = SafePickPlaceEnv(render=False)

In [ ]:
class LMP_wrapper():

  def __init__(self, env, cfg, render=False):
    self.env = env
    self._cfg = cfg
    self.object_names = list(self._cfg['env']['init_objs'])
    
    self._min_xy = np.array(self._cfg['env']['coords']['bottom_left'])
    self._max_xy = np.array(self._cfg['env']['coords']['top_right'])
    self._range_xy = self._max_xy - self._min_xy

    self._table_z = self._cfg['env']['coords']['table_z']
    self.render = render

  def is_obj_visible(self, obj_name):
    return obj_name in self.object_names

  def get_obj_names(self):
    return self.object_names[::]

  def denormalize_xy(self, pos_normalized):
    return pos_normalized * self._range_xy + self._min_xy

  def get_corner_positions(self):
    unit_square = box(0, 0, 1, 1)
    normalized_corners = np.array(list(unit_square.exterior.coords))[:4]
    corners = np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))
    return corners

  def get_side_positions(self):
    side_xs = np.array([0, 0.5, 0.5, 1])
    side_ys = np.array([0.5, 0, 1, 0.5])
    normalized_side_positions = np.c_[side_xs, side_ys]
    side_positions = np.array(([self.denormalize_xy(corner) for corner in normalized_side_positions]))
    return side_positions

  def get_obj_pos(self, obj_name):
    # return the xy position of the object in robot base frame
    return self.env.get_obj_pos(obj_name)[:2]

  def get_obj_position_np(self, obj_name):
    return self.get_pos(obj_name)

  def get_bbox(self, obj_name):
    # return the axis-aligned object bounding box in robot base frame (not in pixels)
    # the format is (min_x, min_y, max_x, max_y)
    bbox = self.env.get_bounding_box(obj_name)
    return bbox

  def get_color(self, obj_name):
    for color, rgb in COLORS.items():
      if color in obj_name:
        return rgb

  def pick_place(self, pick_pos, place_pos):
    pick_pos_xyz = np.r_[pick_pos, [self._table_z]]
    place_pos_xyz = np.r_[place_pos, [self._table_z]]
    pass

  def put_first_on_second(self, arg1, arg2):
    # put the object with obj_name on top of target
    # target can either be another object name, or it can be an x-y position in robot base frame
    pick_pos = self.get_obj_pos(arg1) if isinstance(arg1, str) else arg1
    place_pos = self.get_obj_pos(arg2) if isinstance(arg2, str) else arg2
    self.env.step(action={'pick': pick_pos, 'place': place_pos})

  def get_robot_pos(self):
    # return robot end-effector xy position in robot base frame
    return self.env.get_ee_pos()

  def goto_pos(self, position_xy):
    # move the robot end-effector to the desired xy position while maintaining same z
    ee_xyz = self.env.get_ee_pos()
    position_xyz = np.concatenate([position_xy, ee_xyz[-1]])
    while np.linalg.norm(position_xyz - ee_xyz) > 0.01:
      self.env.movep(position_xyz)
      self.env.step_sim_and_render()
      ee_xyz = self.env.get_ee_pos()

  def follow_traj(self, traj):
    for pos in traj:
      self.goto_pos(pos)

  def get_corner_positions(self):
    normalized_corners = np.array([
        [0, 1],
        [1, 1],
        [0, 0],
        [1, 0]
    ])
    return np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))

  def get_side_positions(self):
    normalized_sides = np.array([
        [0.5, 1],
        [1, 0.5],
        [0.5, 0],
        [0, 0.5]
    ])
    return np.array(([self.denormalize_xy(side) for side in normalized_sides]))

  def get_corner_name(self, pos):
    corner_positions = self.get_corner_positions()
    corner_idx = np.argmin(np.linalg.norm(corner_positions - pos, axis=1))
    return ['top left corner', 'top right corner', 'bottom left corner', 'bottom right corner'][corner_idx]

  def get_side_name(self, pos):
    side_positions = self.get_side_positions()
    side_idx = np.argmin(np.linalg.norm(side_positions - pos, axis=1))
    return ['top side', 'right side', 'bottom side', 'left side'][side_idx]

In [ ]:
prompt_tabletop_ui = '''
# Python 2D robot control script
import numpy as np
from env_utils import put_first_on_second, get_obj_pos, get_obj_names, say, get_corner_name, get_side_name, is_obj_visible, stack_objects_in_order
from plan_utils import parse_obj_name, parse_position, parse_question, transform_shape_pts

objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# place the yellow block on the yellow bowl.
say('Ok - putting the yellow block on the yellow bowl')
put_first_on_second('yellow block', 'yellow bowl')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# which block did you move.
say('I moved the yellow block')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# move the green block to the top right corner.
say('Got it - putting the green block on the top right corner')
corner_pos = parse_position('top right corner')
put_first_on_second('green block', corner_pos)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# stack the blue bowl on the yellow bowl on the green block.
order_bottom_to_top = ['green block', 'yellow block', 'blue bowl']
say(f'Sure - stacking from top to bottom: {", ".join(order_bottom_to_top)}')
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# move the cyan block into its corresponding bowl.
matches = {'cyan block': 'cyan bowl'}
say('Got it - placing the cyan block on the cyan bowl')
for first, second in matches.items():
  put_first_on_second(first, get_obj_pos(second))
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# make a line of blocks on the right side.
say('No problem! Making a line of blocks on the right side')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
line_pts = parse_position(f'a 30cm vertical line on the right with {len(block_names)} points')
for block_name, pt in zip(block_names, line_pts):
  put_first_on_second(block_name, pt)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# put the small banana colored thing in between the blue bowl and green block.
say('Sure thing - putting the yellow block between the blue bowl and the green block')
target_pos = parse_position('a point in the middle betweeen the blue bowl and the green block')
put_first_on_second('yellow block', target_pos)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# can you cut the bowls in half.
say('no, I can only move objects around')
objects = ['yellow block', 'green block', 'yellow bowl', 'gray block', 'gray bowl', 'green bowl']
# stack the blocks on the right side with the gray one on the bottom.
say('Ok. stacking the blocks on the right side with the gray block on the bottom')
right_side = parse_position('the right side')
put_first_on_second('gray block', right_side)
order_bottom_to_top = ['gray block', 'green block', 'yellow block']
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# hide the blue bowl.
bowl_name = np.random.choice(['yellow bowl', 'green bowl'])
say(f'Sounds good! Hiding the blue bowl under the {bowl_name}')
put_first_on_second(bowl_name, 'blue bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the grass-colored bowl to the left.
say('Sure - moving the green bowl left by 10 centimeters')
left_pos = parse_position('a point 10cm left of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# why did you move the red bowl.
say(f'I did not move the red bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# undo that.
say('Sure - moving the green bowl right by 10 centimeters')
left_pos = parse_position('a point 10cm right of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the top most block to the corner closest to the bottom most block.
top_block_name = parse_obj_name('top most block', f'objects = {get_obj_names()}')
bottom_block_name = parse_obj_name('bottom most block', f'objects = {get_obj_names()}')
closest_corner_pos = parse_position(f'the corner closest to the {bottom_block_name}', f'objects = {get_obj_names()}')
say(f'Putting the {top_block_name} on the {get_corner_name(closest_corner_pos)}')
put_first_on_second(top_block_name, closest_corner_pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the brown bowl to the side closest to the green block.
closest_side_position = parse_position('the side closest to the green block')
say(f'Got it - putting the brown bowl on the {get_side_name(closest_side_position)}')
put_first_on_second('brown bowl', closest_side_position)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the green block to the right of the bowl that has the blue block.
bowl_name = parse_obj_name('the bowl that has the blue block', f'objects = {get_obj_names()}')
if bowl_name:
  target_pos = parse_position(f'a point 10cm to the right of the {bowl_name}')
  say(f'No problem - placing the green block to the right of the {bowl_name}')
  put_first_on_second('green block', target_pos)
else:
  say('There are no bowls that has the blue block')
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the other blocks to the bottom corners.
block_names = parse_obj_name('blocks other than the blue block', f'objects = {get_obj_names()}')
corners = parse_position('the bottom corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['pink block', 'gray block', 'orange block']
# move the pinkish colored block on the bottom side.
say('Ok - putting the pink block on the bottom side')
bottom_side_pos = parse_position('the bottom side')
put_first_on_second('pink block', bottom_side_pos)
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# is the blue block to the right of the yellow bowl?
if parse_question('is the blue block to the right of the yellow bowl?', f'objects = {get_obj_names()}'):
  say('yes, there is a blue block to the right of the yellow bow')
else:
  say('no, there is\'t a blue block to the right of the yellow bow')
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# how many yellow objects are there?
n_yellow_objs = parse_question('how many yellow objects are there', f'objects = {get_obj_names()}')
say(f'there are {n_yellow_objs} yellow object')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the left most block to the green bowl.
left_block_name = parse_obj_name('left most block', f'objects = {get_obj_names()}')
say(f'Moving the {left_block_name} on the green bowl')
put_first_on_second(left_block_name, 'green bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the other blocks to different corners.
block_names = parse_obj_name(f'blocks other than the {left_block_name}', f'objects = {get_obj_names()}')
corners = parse_position('the corners')
say(f'Ok - moving the other {len(block_names)} blocks to different corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl.
if parse_question('is the pink block on the green bowl', f'objects = {get_obj_names()}'):
  say('Yes - the pink block is on the green bowl.')
else:
  say('No - the pink block is not on the green bowl.')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl.
left_block_names =  parse_question('what are the blocks left of the green bowl', f'objects = {get_obj_names()}')
if len(left_block_names) > 0:
  say(f'These blocks are left of the green bowl: {", ".join(left_block_names)}')
else:
  say('There are no blocks left of the green bowl')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# imagine that the bowls are different biomes on earth and imagine that the blocks are parts of a building.
say('ok')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# now build a tower in the grasslands.
order_bottom_to_top = ['green bowl', 'blue block', 'green block', 'yellow block']
say('stacking the blocks on the green bowl')
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['yellow block', 'green block', 'yellow bowl', 'gray block', 'gray bowl', 'green bowl']
# show me what happens when the desert gets flooded by the ocean.
say('putting the yellow bowl on the blue bowl')
put_first_on_second('yellow bowl', 'blue bowl')
objects = ['pink block', 'gray block', 'orange block']
# move all blocks 5cm toward the top.
say('Ok - moving all blocks 5cm toward the top')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name in block_names:
  target_pos = parse_position(f'a point 5cm above the {block_name}')
  put_first_on_second(block_name, target_pos)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make a triangle of blocks in the middle.
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
triangle_pts = parse_position(f'a triangle with size 10cm around the middle with {len(block_names)} points')
say('Making a triangle of blocks around the middle of the workspace')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make the triangle smaller.
triangle_pts = transform_shape_pts('scale it by 0.5x', shape_pts=triangle_pts)
say('Making the triangle smaller')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['brown bowl', 'red block', 'brown block', 'red bowl', 'pink bowl', 'pink block']
# put the red block on the farthest bowl.
farthest_bowl_name = parse_obj_name('the bowl farthest from the red block', f'objects = {get_obj_names()}')
say(f'Putting the red block on the {farthest_bowl_name}')
put_first_on_second('red block', farthest_bowl_name)
'''.strip()

prompt_parse_obj_name = '''
import numpy as np
from env_utils import get_obj_pos, parse_position
from utils import get_obj_positions_np

objects = ['blue block', 'cyan block', 'purple bowl', 'gray bowl', 'brown bowl', 'pink block', 'purple block']
# the block closest to the purple bowl.
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=get_obj_pos('purple bowl'))
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the blocks.
ret_val = ['brown block', 'blue block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the brown objects.
ret_val = ['brown bowl', 'brown block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# a fruit that's not the apple
fruit_names = ['banana', 'apple']
for fruit_name in fruit_names:
    if fruit_name != 'apple':
        ret_val = fruit_name
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# blocks above the brown bowl.
block_names = ['blue block', 'cyan block', 'purple block']
brown_bowl_pos = get_obj_pos('brown bowl')
use_block_names = []
for block_name in block_names:
    if get_obj_pos(block_name)[1] > brown_bowl_pos[1]:
        use_block_names.append(block_name)
ret_val = use_block_names
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the blue block.
ret_val = 'blue block'
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the block closest to the bottom right corner.
corner_pos = parse_position('bottom right corner')
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=corner_pos)
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the left most block.
block_names = ['green block', 'brown block', 'blue block']
block_positions = get_obj_positions_np(block_names)
left_block_idx = np.argsort(block_positions[:, 0])[0]
left_block_name = block_names[left_block_idx]
ret_val = left_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the bowl on near the top.
bowl_names = ['brown bowl', 'green bowl', 'blue bowl']
bowl_positions = get_obj_positions_np(bowl_names)
top_bowl_idx = np.argsort(bowl_positions[:, 1])[-1]
top_bowl_name = bowl_names[top_bowl_idx]
ret_val = top_bowl_name
objects = ['yellow bowl', 'purple block', 'yellow block', 'purple bowl', 'pink bowl', 'pink block']
# the third bowl from the right.
bowl_names = ['yellow bowl', 'purple bowl', 'pink bowl']
bowl_positions = get_obj_positions_np(bowl_names)
bowl_idx = np.argsort(bowl_positions[:, 0])[-3]
bowl_name = bowl_names[bowl_idx]
ret_val = bowl_name
'''.strip()

prompt_parse_position = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *
from env_utils import denormalize_xy, parse_obj_name, get_obj_names, get_obj_pos

# a 30cm horizontal line in the middle with 3 points.
middle_pos = denormalize_xy([0.5, 0.5]) 
start_pos = middle_pos + [-0.3/2, 0]
end_pos = middle_pos + [0.3/2, 0]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=3)
ret_val = points
# a 20cm vertical line near the right with 4 points.
middle_pos = denormalize_xy([1, 0.5]) 
start_pos = middle_pos + [0, -0.2/2]
end_pos = middle_pos + [0, 0.2/2]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=4)
ret_val = points
# a diagonal line from the top left to the bottom right corner with 5 points.
top_left_corner = denormalize_xy([0, 1])
bottom_right_corner = denormalize_xy([1, 0])
line = make_line(start=top_left_corner, end=bottom_right_corner)
points = interpolate_pts_on_line(line=line, n=5)
ret_val = points
# a triangle with size 10cm with 3 points.
polygon = make_triangle(size=0.1, center=denormalize_xy([0.5, 0.5]))
points = get_points_from_polygon(polygon)
ret_val = points
# the corner closest to the sun colored block.
block_name = parse_obj_name('the sun colored block', f'objects = {get_obj_names()}')
corner_positions = np.array([denormalize_xy(pos) for pos in [[0, 0], [0, 1], [1, 1], [1, 0]]])
closest_corner_pos = get_closest_point(points=corner_positions, point=get_obj_pos(block_name))
ret_val = closest_corner_pos
# the side farthest from the right most bowl.
bowl_name = parse_obj_name('the right most bowl', f'objects = {get_obj_names()}')
side_positions = np.array([denormalize_xy(pos) for pos in [[0.5, 0], [0.5, 1], [1, 0.5], [0, 0.5]]])
farthest_side_pos = get_farthest_point(points=side_positions, point=get_obj_pos(bowl_name))
ret_val = farthest_side_pos
# a point above the third block from the bottom.
block_name = parse_obj_name('the third block from the bottom', f'objects = {get_obj_names()}')
ret_val = get_obj_pos(block_name) + [0.1, 0]
# a point 10cm left of the bowls.
bowl_names = parse_obj_name('the bowls', f'objects = {get_obj_names()}')
bowl_positions = get_all_object_positions_np(obj_names=bowl_names)
left_obj_pos = bowl_positions[np.argmin(bowl_positions[:, 0])] + [-0.1, 0]
ret_val = left_obj_pos
# the bottom side.
bottom_pos = denormalize_xy([0.5, 0])
ret_val = bottom_pos
# the top corners.
top_left_pos = denormalize_xy([0, 1])
top_right_pos = denormalize_xy([1, 1])
ret_val = [top_left_pos, top_right_pos]
'''.strip()

prompt_parse_question = '''
from utils import get_obj_pos, get_obj_names, parse_obj_name, bbox_contains_pt, is_obj_visible

objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# is the blue block to the right of the yellow bowl?
ret_val = get_obj_pos('blue block')[0] > get_obj_pos('yellow bowl')[0]
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# how many yellow objects are there?
yellow_object_names = parse_obj_name('the yellow objects', f'objects = {get_obj_names()}')
ret_val = len(yellow_object_names)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl?
ret_val = bbox_contains_pt(container_name='green bowl', obj_name='pink block')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
green_bowl_pos = get_obj_pos('green bowl')
left_block_names = []
for block_name in block_names:
  if get_obj_pos(block_name)[0] < green_bowl_pos[0]:
    left_block_names.append(block_name)
ret_val = left_block_names
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# is the sun colored block above the blue bowl?
sun_block_name = parse_obj_name('sun colored block', f'objects = {get_obj_names()}')
sun_block_pos = get_obj_pos(sun_block_name)
blue_bowl_pos = get_obj_pos('blue bowl')
ret_val = sun_block_pos[1] > blue_bowl_pos[1]
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# is the green block below the blue bowl?
ret_val = get_obj_pos('green block')[1] < get_obj_pos('blue bowl')[1]
'''.strip()

prompt_transform_shape_pts = '''
import numpy as np
from utils import get_obj_pos, get_obj_names, parse_position, parse_obj_name

# make it bigger by 1.5.
new_shape_pts = scale_pts_around_centroid_np(shape_pts, scale_x=1.5, scale_y=1.5)
# move it to the right by 10cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0.1, 0])
# move it to the top by 20cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0, 0.2])
# rotate it clockwise by 40 degrees.
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=-np.deg2rad(40))
# rotate by 30 degrees and make it slightly smaller
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=np.deg2rad(30))
new_shape_pts = scale_pts_around_centroid_np(new_shape_pts, scale_x=0.7, scale_y=0.7)
# move it toward the blue block.
block_name = parse_obj_name('the blue block', f'objects = {get_obj_names()}')
block_pos = get_obj_pos(block_name)
mean_delta = np.mean(block_pos - shape_pts, axis=1)
new_shape_pts = translate_pts_np(shape_pts, mean_delta)
'''.strip()

prompt_fgen = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *

from env_utils import get_obj_pos, get_obj_names
from ctrl_utils import put_first_on_second

# define function: total = get_total(xs=numbers).
def get_total(xs):
    return np.sum(xs)

# define function: y = eval_line(x, slope, y_intercept=0).
def eval_line(x, slope, y_intercept):
    return x * slope + y_intercept

# define function: pt = get_pt_to_the_left(pt, dist).
def get_pt_to_the_left(pt, dist):
    return pt + [-dist, 0]

# define function: pt = get_pt_to_the_top(pt, dist).
def get_pt_to_the_top(pt, dist):
    return pt + [0, dist]

# define function line = make_line_by_length(length=x).
def make_line_by_length(length):
  line = LineString([[0, 0], [length, 0]])
  return line

# define function: line = make_vertical_line_by_length(length=x).
def make_vertical_line_by_length(length):
  line = make_line_by_length(length)
  vertical_line = rotate(line, 90)
  return vertical_line

# define function: pt = interpolate_line(line, t=0.5).
def interpolate_line(line, t):
  pt = line.interpolate(t, normalized=True)
  return np.array(pt.coords[0])

# example: scale a line by 2.
line = make_line_by_length(1)
new_shape = scale(line, xfact=2, yfact=2)

# example: put object1 on top of object0.
put_first_on_second('object1', 'object0')

# example: get the position of the first object.
obj_names = get_obj_names()
pos_2d = get_obj_pos(obj_names[0])
'''.strip()

In [ ]:
cfg_tabletop = {
  'lmps': {
    'tabletop_ui': {
      'prompt_text': prompt_tabletop_ui,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': True,
      'debug_mode': False,
      'include_context': True,
      'has_return': False,
      'return_val_name': 'ret_val',
    },
    'parse_obj_name': {
      'prompt_text': prompt_parse_obj_name,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_position': {
      'prompt_text': prompt_parse_position,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_question': {
      'prompt_text': prompt_parse_question,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'transform_shape_pts': {
      'prompt_text': prompt_transform_shape_pts,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'new_shape_pts',
    },
    'fgen': {
      'prompt_text': prompt_fgen,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# define function: ',
      'query_suffix': '.',
      'stop': ['# define', '# example'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
    }
  }
}

lmp_tabletop_coords = {
        'top_left':     (-0.3 + 0.05, -0.2 - 0.05),
        'top_side':     (0,           -0.2 - 0.05),
        'top_right':    (0.3 - 0.05,  -0.2 - 0.05),
        'left_side':    (-0.3 + 0.05, -0.5,      ),
        'middle':       (0,           -0.5,      ),
        'right_side':   (0.3 - 0.05,  -0.5,      ),
        'bottom_left':  (-0.3 + 0.05, -0.8 + 0.05),
        'bottom_side':  (0,           -0.8 + 0.05),
        'bottom_right': (0.3 - 0.05,  -0.8 + 0.05),
        'table_z':       0.0,
      }

In [ ]:
def setup_LMP(env, cfg_tabletop, backend):
    cfg_tabletop = copy.deepcopy(cfg_tabletop)
    cfg_tabletop['env'] = dict()
    cfg_tabletop['env']['init_objs'] = list(env.obj_name_to_id.keys())
    cfg_tabletop['env']['coords'] = lmp_tabletop_coords
    LMP_env = LMP_wrapper(env, cfg_tabletop)

    fixed_vars = {'np': np}
    fixed_vars.update({
        name: eval(name)
        for name in shapely.geometry.__all__ + shapely.affinity.__all__
    })
    variable_vars = {
        k: getattr(LMP_env, k)
        for k in [
            'get_bbox', 'get_obj_pos', 'get_color', 'is_obj_visible', 'denormalize_xy',
            'put_first_on_second', 'get_obj_names',
            'get_corner_name', 'get_side_name',
        ]
    }

    # 発話を記録する（質問タスクの正誤判定に使う）
    said = []
    def say(msg):
        said.append(str(msg))
        _log(f'robot says: {msg}')
    variable_vars['say'] = say

    lmp_fgen = LMPFGen(cfg_tabletop['lmps']['fgen'], fixed_vars, variable_vars, backend)
    variable_vars.update({
        k: LMP(k, cfg_tabletop['lmps'][k], lmp_fgen, fixed_vars, variable_vars, backend)
        for k in ['parse_obj_name', 'parse_position', 'parse_question', 'transform_shape_pts']
    })
    lmp_tabletop_ui = LMP(
        'tabletop_ui', cfg_tabletop['lmps']['tabletop_ui'], lmp_fgen, fixed_vars, variable_vars, backend
    )
    return lmp_tabletop_ui, said

In [ ]:
CORNERS = ['top left corner', 'top right corner', 'bottom left corner', 'bottom right corner']
COLOR_ALIASES = {
    'yellow': ['sun-colored', 'banana-colored'],
    'green':  ['grass-colored', 'forest-colored'],
    'blue':   ['ocean-colored', 'sky-colored'],
    'red':    ['strawberry-colored', 'fire-truck-colored'],
    'orange': ['carrot-colored', 'pumpkin-colored'],
    'purple': ['grape-colored', 'eggplant-colored'],
    'pink':   ['bubblegum-colored', 'flamingo-colored'],
    'cyan':   ['turquoise-colored', 'teal'],
    'brown':  ['chocolate-colored', 'coffee-colored'],
    'gray':   ['ash-colored', 'concrete-colored'],
}
NUM_WORDS = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten']


def _pos(name):
    return env.get_obj_pos(name)


def _dist_xy(a, b):
    return float(np.linalg.norm(np.asarray(a)[:2] - np.asarray(b)[:2]))


def _color(name):
    return name.split(' ')[0]


# 各テンプレートは (指示文, 判定関数) を返す。使えない配置なら None を返す。
# 判定関数は check(said) の形で、実行後の env と発話リスト said を見て True/False を返す。

def tpl_block_on_bowl(rng, blocks, bowls):
    b, w = rng.choice(blocks), rng.choice(bowls)
    fmt = rng.choice(['put the {b} on the {w}', 'place the {b} in the {w}',
                      'move the {b} into the {w}', 'pick up the {b} and drop it in the {w}'])
    return fmt.format(b=b, w=w), lambda said: env.on_top_of(b, w)


def tpl_color_alias(rng, blocks, bowls):
    b, w = rng.choice(blocks), rng.choice(bowls)
    alias = rng.choice(COLOR_ALIASES[_color(b)])
    return f'put the {alias} block on the {w}', lambda said: env.on_top_of(b, w)


def tpl_corner(rng, blocks, bowls):
    b, corner = rng.choice(blocks), rng.choice(CORNERS)
    return f'move the {b} to the {corner}', lambda said: env.on_top_of(b, corner)


def tpl_closest_bowl(rng, blocks, bowls):
    if len(bowls) < 2:
        return None
    b = rng.choice(blocks)
    ranked = sorted((_dist_xy(_pos(b), _pos(w)), w) for w in bowls)
    if ranked[1][0] - ranked[0][0] < 0.03:   # 紛らわしい配置は使わない
        return None
    target = ranked[0][1]
    return f'put the {b} in the bowl closest to it', lambda said: env.on_top_of(b, target)


def tpl_extreme_block(rng, blocks, bowls):
    if len(blocks) < 2:
        return None
    # (表現, 座標軸, 符号)：符号付きで小さい順に並べた先頭が正解
    desc, axis, sign = rng.choice([('left most', 0, 1), ('right most', 0, -1),
                                   ('top most', 1, -1), ('bottom most', 1, 1)])
    ranked = sorted((sign * _pos(b)[axis], b) for b in blocks)
    if ranked[1][0] - ranked[0][0] < 0.03:
        return None
    target, w = ranked[0][1], rng.choice(bowls)
    return f'put the {desc} block on the {w}', lambda said: env.on_top_of(target, w)


def tpl_side_of(rng, blocks, bowls):
    b, w = rng.choice(blocks), rng.choice(bowls)
    side = rng.choice(['left', 'right'])

    def check(said):
        (bx, by), (wx, wy) = _pos(b)[:2], _pos(w)[:2]
        ok_side = bx < wx - 0.04 if side == 'left' else bx > wx + 0.04
        return bool(ok_side and abs(by - wy) < 0.08)
    return f'put the {b} to the {side} of the {w}', check


def tpl_stack(rng, blocks, bowls):
    if len(blocks) < 2:
        return None
    w = rng.choice(bowls)

    def check(said):
        wp = _pos(w)
        return all(_dist_xy(_pos(b), wp) < 0.07 and _pos(b)[2] > wp[2] for b in blocks)
    return f'stack all the blocks on the {w}', check


def tpl_matching(rng, blocks, bowls):
    pairs = [(b, f'{_color(b)} bowl') for b in blocks if f'{_color(b)} bowl' in bowls]
    if not pairs:
        return None
    instruction = rng.choice(['put the blocks in the bowls with matching colors',
                              'put each block in the bowl of the same color'])
    return instruction, lambda said: all(env.on_top_of(b, w) for b, w in pairs)


def tpl_count(rng, blocks, bowls):
    kind = rng.choice(['blocks', 'bowls'])
    n = len(blocks) if kind == 'blocks' else len(bowls)

    def check(said):
        text = ' '.join(said).lower()
        return bool(re.search(rf'\b({n}|{NUM_WORDS[n]})\b', text))
    return f'how many {kind} are there?', check


TEMPLATES = {
    'block_on_bowl': tpl_block_on_bowl,
    'color_alias': tpl_color_alias,
    'corner': tpl_corner,
    'closest_bowl': tpl_closest_bowl,
    'extreme_block': tpl_extreme_block,
    'side_of': tpl_side_of,
    'stack': tpl_stack,
    'matching': tpl_matching,
    'count': tpl_count,
}
TRAIN_TEMPLATES = [t for t in TEMPLATES if t not in HOLDOUT_TEMPLATES]


def sample_task(rng, blocks, bowls, allowed_templates):
    """使えるテンプレートが見つかるまでランダムに試す。"""
    names = list(allowed_templates)
    rng.shuffle(names)
    for name in names:
        task = TEMPLATES[name](rng, blocks, bowls)
        if task is not None:
            return name, task[0], task[1]
    raise RuntimeError('どのテンプレートも使えないシーンでした')

In [ ]:
def placement_is_feasible(n_objects, max_tries=5000):
    """env.reset と同じ手順で物体の配置を試し、置き場所が見つかるかを調べる。
    乱数の状態は元に戻すので、この後の env.reset はまったく同じ配置になる。"""
    state = np.random.get_state()
    placed = np.zeros((0, 2))
    tries = 0
    try:
        for _ in range(n_objects):
            while True:
                tries += 1
                if tries > max_tries:
                    return False
                x = np.random.uniform(BOUNDS[0, 0] + 0.1, BOUNDS[0, 1] - 0.1)
                y = np.random.uniform(BOUNDS[1, 0] + 0.1, BOUNDS[1, 1] - 0.1)
                p = np.float32([x, y]).reshape(1, 2)
                if len(placed) == 0 or np.min(np.linalg.norm(placed - p, axis=1)) > 0.15:
                    placed = np.concatenate((placed, p), axis=0)
                    break
        return True
    finally:
        np.random.set_state(state)


def run_episode(seed, backend, allowed_templates):
    rng = random.Random(seed)
    np.random.seed(seed)  # env.reset の物体配置も seed で決まるようにする

    # 置き場所が詰まらない組み合わせになるまで物体を選び直す
    while True:
        blocks = rng.sample(ALL_BLOCKS, rng.randint(2, 4))
        bowls = rng.sample(ALL_BOWLS, rng.randint(1, 4))
        if placement_is_feasible(len(blocks) + len(bowls)):
            break
    env.reset(blocks + bowls)

    template, instruction, check = sample_task(rng, blocks, bowls, allowed_templates)
    lmp_tabletop_ui, said = setup_LMP(env, cfg_tabletop, backend)

    RECORDER.reset()
    t0 = time.time()
    error = None
    try:
        lmp_tabletop_ui(instruction, f'objects = {env.object_list}')
    except genai_errors.APIError:
        raise  # API の上限などはループ側で処理する
    except SimTimeout as e:
        error = f'SimTimeout: {e}'
    except Exception as e:
        error = f'{type(e).__name__}: {e}'

    success = error is None and bool(check(said))
    return {
        'seed': seed,
        'template': template,
        'instruction': instruction,
        'objects': list(env.object_list),
        'success': success,
        'error': error,
        'said': said,
        'calls': list(RECORDER.calls),
        'llm_time': sum(c['latency'] for c in RECORDER.calls),
        'total_time': time.time() - t0,
    }

## 3. 学習済みの生徒モデルを読み込む

学習ノートブックでは GPU のメモリ量でベースモデル（7B / 1.5B）を選んでいたので、別の GPU で開き直すと違うモデルが選ばれてしまいます。ここでは **アダプタの設定ファイル（`adapter_config.json`）に書かれたベースモデル** を読み込むので、学習時と必ず同じモデルになります。

生徒への入力形式（`build_student_messages`）と `StudentBackend` は学習ノートブックと同じです。

In [ ]:
import transformers
from packaging import version
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

with open(f'{ADAPTER_DIR}/adapter_config.json', encoding='utf-8') as f:
    STUDENT_BASE_MODEL = json.load(f)['base_model_name_or_path']

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
# 推論だけなので、bf16 が使えない GPU（T4）でも fp16 で読み込めば十分
LOAD_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'GPU: {gpu_name} ({gpu_mem_gb:.0f}GB) / 生徒のベースモデル: {STUDENT_BASE_MODEL} / {LOAD_DTYPE}')
if '7B' in STUDENT_BASE_MODEL and gpu_mem_gb < 20:
    print('⚠️ 7B モデルはこの GPU ではメモリが足りない可能性があります（A100 を推奨）')

dtype_kw = 'dtype' if version.parse(transformers.__version__) >= version.parse('4.56.0') else 'torch_dtype'
base_model = AutoModelForCausalLM.from_pretrained(STUDENT_BASE_MODEL, device_map={'': 0}, **{dtype_kw: LOAD_DTYPE})
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# 動画に表示する名前（例: Qwen2.5-Coder-7B-Instruct + LoRA）
STUDENT_LABEL = STUDENT_BASE_MODEL.split('/')[-1] + ' + LoRA'

In [ ]:
STUDENT_SYSTEM_PROMPT = (
    'You are a code-writing policy for a tabletop robot. '
    'Given the LMP name, the context and an instruction comment, '
    'write only the Python code that should follow. No explanations.'
)


def build_student_messages(lmp, context, query, history=''):
    parts = [f'# LMP: {lmp}']
    if history.strip():
        parts.append(history.strip())
    if context.strip():
        parts.append(context.strip())
    parts.append(query)
    return [
        {'role': 'system', 'content': STUDENT_SYSTEM_PROMPT},
        {'role': 'user', 'content': '\n'.join(parts)},
    ]


class StudentBackend:
    """生徒バックエンド：ファインチューニングした小型モデルで、短い入力からコードを書く。"""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    @torch.no_grad()
    def __call__(self, lmp_name, full_prompt, context, query, history, stop, cfg):
        msgs = build_student_messages(lmp_name, context, query, history)
        text = self.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inputs = self.tokenizer(text, return_tensors='pt', add_special_tokens=False).to(self.model.device)
        out = self.model.generate(
            **inputs,
            max_new_tokens=cfg['max_tokens'],
            do_sample=False,
            use_cache=True,
            max_time=60,                    # 1 回の生成は最大 60 秒で打ち切る
            pad_token_id=self.tokenizer.pad_token_id,
        )
        completion = self.tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return _postprocess(completion, query, stop)


student_backend = StudentBackend(model, tokenizer)

## 4. 動画にするタスクを決める

`DEMO_TASKS` に、動画に入れるタスクを順番に並べます。

| 種類 | 書き方 | 成否 |
|---|---|---|
| テンプレート | `{'template': 'stack', 'seed': 100060}` | 自動判定（学習ノートブックの判定関数） |
| 自由な指示 | `{'instruction': '...', 'objects': [...], 'seed': 7}` | 自動判定なし（動画を見て判断） |

- テンプレートのタスクは、`seed` から順に探して **そのテンプレートが使えるシーン** を作ります（例：`matching` は同じ色のブロックとボウルがあるシーンが必要）。seed が同じなら毎回同じシーン・同じ指示になるので、教師と生徒を同じ条件で比べられます
- seed は評価と同じ **100000 番台**（学習に使っていない seed）にしています
- 最後の 2 つは学習テンプレートに無い言い回しで、生徒がどこまで汎化するかを見るためのものです。失敗してもそれ自体が結果です

**動画の長さ**：1 タスクあたりおよそ 5〜15 秒（2 倍速）です。全体が 1〜3 分に収まるよう、最後のセルで長さを確認して、長すぎるときはタスクを減らしてください。

In [ ]:
DEMO_TASKS = [
    {'template': 'block_on_bowl', 'seed': 100000},
    {'template': 'color_alias',   'seed': 100010},
    {'template': 'corner',        'seed': 100020},
    {'template': 'closest_bowl',  'seed': 100030},
    {'template': 'extreme_block', 'seed': 100040},
    {'template': 'side_of',       'seed': 100050},
    {'template': 'matching',      'seed': 100060},
    {'template': 'stack',         'seed': 100070},
    {'template': 'count',         'seed': 100080},
    # --- 学習テンプレートに無い指示（汎化のテスト）---
    {'instruction': 'put the red block in the bowl farthest from it',
     'objects': ['red block', 'blue block', 'green block', 'blue bowl', 'yellow bowl'], 'seed': 7},
    {'instruction': 'put the blocks in a horizontal line near the bottom side',
     'objects': ['red block', 'blue block', 'yellow block', 'green bowl'], 'seed': 8},
]

TEMPLATE_JP = {
    'block_on_bowl': 'ブロックをボウルに入れる',
    'color_alias':   '色の言い換えを理解する',
    'corner':        'ブロックを角へ動かす',
    'closest_bowl':  '一番近いボウルを選ぶ',
    'extreme_block': '一番端のブロックを選ぶ',
    'side_of':       'ボウルの左／右に置く',
    'stack':         'ブロックを積み上げる',
    'matching':      '同じ色のボウルに入れる',
    'count':         '質問に答える（数を数える）',
    'free':          '自由な指示（学習テンプレート外）',
}

## 5. 1 タスクを実行して録画する

`record_task(task, backend)` は、学習ノートブックの `run_episode` と同じ手順でシーンを作って CaP を実行し、**録画した画像・生成コード・成否** を返します。違いは次の点です。

- 録画を ON にし、画質を上げる（360×360、12 枚/秒）
- 実行前の最初の画面も記録する（指示を読む時間を動画に作るため）
- 失敗したタスクが動画を長くしすぎないよう、1 タスクのシミュレーションは最大 45 秒で打ち切る

In [ ]:
env.render = True
env.high_res = True            # 360x360
env.high_frame_rate = True     # 40 ステップごと（1/480 秒 × 40 = 1/12 秒）に 1 枚
SafePickPlaceEnv.MAX_SIM_STEPS_PER_EPISODE = 480 * 45   # 1 タスク最大 45 秒（シミュレーション時間）

SIM_FPS = 480 / (40 if env.high_frame_rate else 60)    # 録画の 1 秒あたりの枚数（シミュレーション時間）
PLAYBACK_FPS = SIM_FPS * PLAYBACK_SPEED
print(f'録画 {SIM_FPS:.0f} 枚/秒 → 書き出し {PLAYBACK_FPS:.0f} fps = {PLAYBACK_SPEED:g} 倍速')


def _make_scene(task):
    """task からシーンを作り、(テンプレート名, 指示文, 判定関数 or None, seed) を返す。"""
    if 'template' not in task:
        np.random.seed(task['seed'])
        env.reset(list(task['objects']))
        return 'free', task['instruction'], None, task['seed']

    for seed in range(task['seed'], task['seed'] + 200):
        rng = random.Random(seed)
        np.random.seed(seed)
        while True:
            blocks = rng.sample(ALL_BLOCKS, rng.randint(2, 4))
            bowls = rng.sample(ALL_BOWLS, rng.randint(1, 4))
            if placement_is_feasible(len(blocks) + len(bowls)):
                break
        env.reset(blocks + bowls)
        made = TEMPLATES[task['template']](rng, blocks, bowls)
        if made is not None:
            return task['template'], made[0], made[1], seed
    raise RuntimeError(f"{task['template']} が使えるシーンが見つかりませんでした")


def record_task(task, backend):
    template, instruction, check, seed = _make_scene(task)
    lmp_tabletop_ui, said = setup_LMP(env, cfg_tabletop, backend)

    env.cache_video = []
    first_frame = env.get_camera_image()
    RECORDER.reset()
    t0 = time.time()
    error = None
    try:
        lmp_tabletop_ui(instruction, f'objects = {env.object_list}')
    except SimTimeout as e:
        error = f'SimTimeout: {e}'
    except Exception as e:      # 生成コードのエラー・API エラーなども結果として記録する
        error = f'{type(e).__name__}: {e}'

    if check is None:
        success = None          # 自由な指示は自動判定しない
    else:
        success = error is None and bool(check(said))
    return {
        'seed': seed,
        'template': template,
        'instruction': instruction,
        'objects': list(env.object_list),
        'success': success,
        'error': error,
        'said': list(said),
        'code': [{'lmp': c['lmp'], 'query': c['query'], 'completion': c['completion']} for c in RECORDER.calls],
        'llm_time': round(sum(c['latency'] for c in RECORDER.calls), 2),
        'total_time': round(time.time() - t0, 2),
        'frames': [first_frame] + env.cache_video + [env.get_camera_image()],
    }

## 6. 動画のレイアウト（1280×720）

```
┌────────────────────────────────────────────────────────┐
│ [3/11] 一番近いボウルを選ぶ                              │
│ "put the red block in the bowl closest to it"          │
├───────────────────────┬────────────────────────────────┤
│                       │ 生徒が生成したコード（tabletop_ui）│
│   シミュレーション映像  │ say('...')                      │
│                       │ put_first_on_second(...)        │
│      （終了時に        │                                 │
│     成功／失敗を表示）  │ 下位 LMP: parse_obj_name ×1 ...  │
│                       │ robot says: ...                 │
├───────────────────────┴────────────────────────────────┤
│ 生徒: Qwen2.5-Coder-7B-Instruct + LoRA | 2 倍速再生      │
└────────────────────────────────────────────────────────┘
```

教師と比較するときは、コード欄の代わりに右側に教師の映像を並べます。文字は Pillow で描くので、日本語も表示できます。

In [ ]:
import glob
import textwrap
from functools import lru_cache
from collections import Counter
import matplotlib
from PIL import Image, ImageDraw, ImageFont

VIDEO_W, VIDEO_H = 1280, 720
BG, PANEL = (22, 24, 30), (38, 41, 50)
TEXT, DIM, ACCENT = (238, 238, 238), (165, 170, 182), (110, 170, 255)
OK_COLOR, NG_COLOR, NA_COLOR = (46, 160, 90), (205, 60, 60), (110, 110, 120)


def _find_font(patterns):
    for p in patterns:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    return None


JP_FONT = _find_font(['/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc', '/usr/share/fonts/**/NotoSansCJK*Regular*'])
JP_BOLD = _find_font(['/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc', '/usr/share/fonts/**/NotoSansCJK*Bold*']) or JP_FONT
MONO_FONT = os.path.join(matplotlib.get_data_path(), 'fonts/ttf/DejaVuSansMono.ttf')
if JP_FONT is None:
    print('⚠️ 日本語フォントが見つかりません（日本語が表示されない可能性があります）')


@lru_cache(maxsize=None)
def font(size, bold=False, mono=False):
    path = MONO_FONT if mono else (JP_BOLD if bold else JP_FONT)
    try:
        return ImageFont.truetype(path, size)
    except Exception:
        return ImageFont.load_default()


def _fit_text(draw, text, max_w, size, bold=False, min_size=16):
    """max_w に収まるまでフォントを小さくする。"""
    while size > min_size and draw.textlength(text, font=font(size, bold)) > max_w:
        size -= 1
    return font(size, bold)


def _to_pil(frame, size):
    return Image.fromarray(np.asarray(frame, dtype=np.uint8)[..., :3]).resize((size, size), Image.BILINEAR)


def _code_lines(ep, width=50, max_lines=17):
    """tabletop_ui が生成したコードを、パネルに収まるよう折り返す。"""
    top = [c['completion'] for c in ep['code'] if c['lmp'] == 'tabletop_ui']
    src = top[0] if top else '(コードが生成されませんでした)'
    lines = []
    for raw in src.split('\n'):
        lines += textwrap.wrap(raw, width=width, subsequent_indent='    ') or ['']
    if len(lines) > max_lines:
        lines = lines[:max_lines - 1] + ['...']
    return lines


def _banner(draw, box, ep):
    """映像の下部に成功／失敗を表示する。"""
    x0, y0, x1, y1 = box
    if ep['success'] is None:
        color, label = NA_COLOR, '自動判定なし（映像で確認）'
    elif ep['success']:
        color, label = OK_COLOR, '○ 成功'
    else:
        color, label = NG_COLOR, '× 失敗'
    h = 64 if (ep['error'] and ep['success'] is False) else 48
    draw.rectangle([x0, y1 - h, x1, y1], fill=color)
    draw.text((x0 + 16, y1 - h + 6), label, font=font(28, bold=True), fill=(255, 255, 255))
    if ep['error'] and ep['success'] is False:
        msg = ep['error'].split('\n')[0]
        msg = msg if len(msg) < 60 else msg[:57] + '...'
        draw.text((x0 + 16, y1 - 22), msg, font=font(14, mono=True), fill=(255, 235, 235))


def _header(draw, idx, total, ep):
    draw.rectangle([0, 0, VIDEO_W, 92], fill=PANEL)
    tag = TEMPLATE_JP.get(ep['template'], ep['template'])
    if ep['template'] in HOLDOUT_TEMPLATES:
        tag += '（学習に使っていないテンプレート）'
    draw.text((30, 10), f'[{idx}/{total}]  {tag}', font=font(22), fill=ACCENT)
    instr = f'"{ep["instruction"]}"'
    draw.text((30, 40), instr, font=_fit_text(draw, instr, VIDEO_W - 60, 32, bold=True), fill=TEXT)


def _footer(draw, text):
    draw.text((30, VIDEO_H - 32), text, font=font(18), fill=DIM)


def make_clip_layout(eps, idx, total):
    """1 つのクリップの (背景画像, 映像を貼る位置のリスト) を作る。eps は [生徒] or [生徒, 教師]。"""
    img = Image.new('RGB', (VIDEO_W, VIDEO_H), BG)
    d = ImageDraw.Draw(img)
    _header(d, idx, total, eps[0])
    speed = f'{PLAYBACK_SPEED:g} 倍速再生（シミュレーション時間比）'

    if len(eps) == 1:
        ep = eps[0]
        slots = [(30, 104, 576)]
        x0, y0, x1, y1 = 636, 104, VIDEO_W - 30, 680
        d.rectangle([x0, y0, x1, y1], fill=PANEL)
        d.text((x0 + 18, y0 + 12), '生徒が生成したコード（tabletop_ui）', font=font(20, bold=True), fill=ACCENT)
        y = y0 + 48
        for line in _code_lines(ep):
            d.text((x0 + 18, y), line, font=font(16, mono=True), fill=TEXT)
            y += 22
        sub = Counter(c['lmp'] for c in ep['code'] if c['lmp'] != 'tabletop_ui')
        sub_txt = '、'.join(f'{k} ×{v}' for k, v in sub.items()) or 'なし'
        info = [f'下位 LMP の呼び出し: {sub_txt}',
                f'コード生成の合計時間: {ep["llm_time"]:.1f} 秒']
        if ep['said']:
            said = 'robot says: ' + ' / '.join(ep['said'])
            info.insert(0, said if len(said) <= 62 else said[:59] + '...')
        y = y1 - 16 - 26 * len(info)
        for line in info:
            d.text((x0 + 18, y), line, font=_fit_text(d, line, x1 - x0 - 36, 18), fill=DIM)
            y += 26
        _footer(d, f'生徒: {STUDENT_LABEL}  |  {speed}')
    else:
        slots = [(40, 132, 548), (VIDEO_W - 40 - 548, 132, 548)]
        labels = [f'生徒: {STUDENT_LABEL}（短い入力のみ）', f'教師: {MODEL_NAME}（few-shot プロンプト）']
        for (x, y, s), label, ep in zip(slots, labels, eps):
            d.text((x, 100), label, font=_fit_text(d, label, s - 150, 20, bold=True), fill=ACCENT)
            t = f'生成 {ep["llm_time"]:.1f} 秒'
            d.text((x + s - d.textlength(t, font=font(18)), 102), t, font=font(18), fill=DIM)
        _footer(d, f'同じシーン・同じ指示で比較  |  {speed}')
    return img, slots


def clip_frames(eps, idx, total, hold_start=1.5, hold_end=2.0):
    """クリップの全フレームを順に返す（最初に指示を読む時間、最後に結果を表示する時間を入れる）。"""
    base, slots = make_clip_layout(eps, idx, total)
    seqs = [ep['frames'] for ep in eps]
    n = max(len(s) for s in seqs)
    n_start, n_end = int(hold_start * PLAYBACK_FPS), int(hold_end * PLAYBACK_FPS)

    def compose(t, with_banner):
        img = base.copy()
        d = ImageDraw.Draw(img)
        for (x, y, s), seq, ep in zip(slots, seqs, eps):
            img.paste(_to_pil(seq[min(max(t, 0), len(seq) - 1)], s), (x, y))
            if with_banner:
                _banner(d, (x, y, x + s - 1, y + s - 1), ep)
        return np.asarray(img)

    for t in range(-n_start, n):
        yield compose(t, False)
    last = compose(n - 1, True)
    for _ in range(n_end):
        yield last


def open_writer(path):
    return imageio.get_writer(path, fps=PLAYBACK_FPS, codec='libx264', quality=8,
                              pixelformat='yuv420p', macro_block_size=16)

## 7. すべてのタスクを実行して、クリップを作る

タスクごとに `clips_<実験名>/NN_<テンプレート>.mp4` を作り、実行記録を `demo_video_<実験名>.jsonl` に追記します。作成済みのクリップはスキップします（作り直すなら設定の `OVERWRITE_CLIPS` を ON に）。

生成されたコードはこのセルの出力にも表示されます。

In [ ]:
def _clip_id(i, task):
    return f"{i:02d}_{task.get('template', 'free')}"


def _load_log():
    """クリップごとの最新の記録（同じ設定で作ったもの）。"""
    latest = {}
    for e in read_jsonl(DEMO_VIDEO_LOG):
        if e.get('compare') == COMPARE_WITH_TEACHER:
            latest[e['clip_id']] = e
    return latest


VERBOSE = True
agents = [('student', student_backend)] + ([('teacher', gemini_backend)] if COMPARE_WITH_TEACHER else [])
log = _load_log()
for i, task in enumerate(DEMO_TASKS, start=1):
    cid = _clip_id(i, task)
    path = f'{CLIP_DIR}/{cid}.mp4'
    if not OVERWRITE_CLIPS and os.path.exists(path) and cid in log:
        print(f'[{cid}] 作成済みなのでスキップ')
        continue

    print(f'\n========== [{cid}] ==========')
    eps = []
    for name, backend in agents:
        ep = record_task(task, backend)
        ep['agent'] = name
        eps.append(ep)
        mark = {True: '成功', False: '失敗', None: '判定なし'}[ep['success']]
        print(f"--> {name}: {mark} / {ep['instruction']} / 生成 {ep['llm_time']:.1f} 秒 {ep['error'] or ''}")

    writer = open_writer(path)
    for frame in clip_frames(eps, i, len(DEMO_TASKS)):
        writer.append_data(frame)
    writer.close()

    for ep in eps:
        rec = {k: v for k, v in ep.items() if k != 'frames'}
        rec.update({'experiment': EXPERIMENT_NAME, 'clip_id': cid, 'clip': path, 'compare': COMPARE_WITH_TEACHER,
                    'student_model': STUDENT_LABEL, 'time': time.strftime('%Y-%m-%d %H:%M:%S')})
        append_jsonl(DEMO_VIDEO_LOG, rec)
    print('保存しました:', path)

### クリップを 1 本確認する

最後に作ったクリップ（または `CHECK_CLIP` で指定したもの）を Notebook 上で再生します。

In [ ]:
CHECK_CLIP = ''  #@param {type:"string"}
_path = f'{CLIP_DIR}/{CHECK_CLIP}.mp4' if CHECK_CLIP else sorted(glob.glob(f'{CLIP_DIR}/*.mp4'))[-1]
with open(_path, 'rb') as f:
    _b64 = base64.b64encode(f.read()).decode()
display(HTML(f'<video width="720" controls autoplay loop muted><source src="data:video/mp4;base64,{_b64}" type="video/mp4"></video>'))
print(_path)

## 8. 1 本の動画にまとめる

**タイトル画面（4 秒）→ 各タスクのクリップ → 結果まとめ画面（5 秒）** の順につないで `FINAL_VIDEO_PATH` に保存し、長さが 1〜3 分に収まっているか確認します。結果の一覧はレポート用に CSV にも保存します。

In [ ]:
def text_card(lines, seconds):
    """lines: [(文字列, サイズ, 太字?, 色), ...] を中央寄せで並べた静止画面。"""
    img = Image.new('RGB', (VIDEO_W, VIDEO_H), BG)
    d = ImageDraw.Draw(img)
    total_h = sum(int(size * 1.6) for _, size, _, _ in lines)
    y = (VIDEO_H - total_h) // 2
    for text, size, bold, color in lines:
        f = _fit_text(d, text, VIDEO_W - 120, size, bold)
        d.text(((VIDEO_W - d.textlength(text, font=f)) / 2, y), text, font=f, fill=color)
        y += int(size * 1.6)
    frame = np.asarray(img)
    return [frame] * int(seconds * PLAYBACK_FPS)


_all_logs = [e for e in read_jsonl(DEMO_VIDEO_LOG) if e.get('compare') == COMPARE_WITH_TEACHER]
rows = []
for i, task in enumerate(DEMO_TASKS, start=1):
    cid = _clip_id(i, task)
    entries = [e for e in _all_logs if e['clip_id'] == cid]   # 生徒・教師それぞれ最新の記録を使う
    for agent in [a for a, _ in agents]:
        mine = [e for e in entries if e['agent'] == agent]
        if mine:
            e = mine[-1]
            rows.append({'clip_id': cid, 'agent': agent, 'template': e['template'], 'instruction': e['instruction'],
                         'success': e['success'], 'error': e['error'], 'llm_time': e['llm_time'], 'said': ' / '.join(e['said'])})
df_demo = pd.DataFrame(rows)
display(df_demo)
df_demo.to_csv(f'{REPORT_DIR}/demo_video_{EXPERIMENT_NAME}' + ('_compare' if COMPARE_WITH_TEACHER else '') + '.csv',
               index=False, encoding='utf-8-sig')

# --- タイトル画面 ---
title = text_card([
    (f'PAI最終課題_{OMNI_ACCOUNT}', 26, False, DIM),
    ('Code as Policies の蒸留', 52, True, TEXT),
    ('Gemini（教師）が書いたロボット制御コードで小型モデル（生徒）を LoRA 学習', 26, False, TEXT),
    (f'生徒: {STUDENT_LABEL}  —  few-shot 例なしの短い入力からコードを生成して実行', 22, False, ACCENT),
    ('PyBullet シミュレーション（UR5e アーム + Robotiq 2F-85 グリッパー）', 22, False, DIM),
    (f'※ 動画は {PLAYBACK_SPEED:g} 倍速再生です', 26, True, (255, 200, 90)),
], seconds=4)

# --- 結果まとめ画面 ---
summary_lines = [('結果まとめ', 44, True, TEXT)]
for agent, label in [('student', f'生徒（{STUDENT_LABEL}）'), ('teacher', f'教師（{MODEL_NAME}）')]:
    sub = df_demo[(df_demo.agent == agent) & df_demo.success.notna()] if len(df_demo) else df_demo
    if len(sub):
        summary_lines.append((f'{label}: 自動判定 {len(sub)} タスク中 {int(sub.success.sum())} 成功', 28, False, ACCENT))
for _, r in df_demo[df_demo.agent == 'student'].iterrows():
    mark = {True: '○', False: '×'}.get(r['success'], '－')
    summary_lines.append((f"{mark}  {TEMPLATE_JP.get(r['template'], r['template'])}", 20, False, DIM))
ending = text_card(summary_lines, seconds=5)

# --- つなぐ ---
writer = open_writer(FINAL_VIDEO_PATH)
n_frames = 0
for frame in title:
    writer.append_data(frame); n_frames += 1
for i, task in enumerate(DEMO_TASKS, start=1):
    path = f'{CLIP_DIR}/{_clip_id(i, task)}.mp4'
    if not os.path.exists(path):
        print('⚠️ クリップがありません（スキップ）:', path)
        continue
    reader = imageio.get_reader(path)
    for frame in reader:
        writer.append_data(frame); n_frames += 1
    reader.close()
for frame in ending:
    writer.append_data(frame); n_frames += 1
writer.close()

duration = n_frames / PLAYBACK_FPS
print(f'保存しました: {FINAL_VIDEO_PATH}')
print(f'長さ: {round(duration)} 秒（{round(duration) // 60} 分 {round(duration) % 60} 秒） / '
      f'{os.path.getsize(FINAL_VIDEO_PATH) / 1e6:.1f} MB')
if duration < 60:
    print('⚠️ 1 分未満です。DEMO_TASKS にタスクを足してください。')
elif duration > 180:
    print('⚠️ 3 分を超えています。DEMO_TASKS からタスクを減らしてください（2 倍速より速くするのは課題の条件外）。')

## 9. YouTube にアップロードする

保存した mp4 を Drive からダウンロードして（または下のセルで直接ダウンロードして）YouTube にアップロードします。

- **タイトル**：`PAI最終課題_<omnicampusアカウント名>`
- **公開設定**：限定公開以上（全体公開だと加点）
- 「子ども向け」・「年齢制限」は **いいえ**、「インスタントプレミア公開」には **チェックしない**
- 概要欄に **倍速再生であること** を書く（下のセルで文例を出力します）

In [ ]:
description = f"""PAI最終課題_{OMNI_ACCOUNT}

Code as Policies（Liang et al., 2022）の卓上マニピュレーションタスクを題材に、
Gemini（{MODEL_NAME}）を教師として作ったロボット制御コードで、
小型のコード生成モデル {STUDENT_LABEL} をファインチューニング（蒸留）しました。
動画では、生徒モデルが few-shot 例なしの短い入力からコードを生成し、
PyBullet シミュレーション上の UR5e アームを動かす様子を示しています。

※ 動画は {PLAYBACK_SPEED:g} 倍速再生です（シミュレーション時間比）。

コード: https://github.com/<ユーザー名>/<リポジトリ名>
"""
print(description)

try:
    from google.colab import files
    files.download(FINAL_VIDEO_PATH)
except Exception:
    print('Drive から直接ダウンロードしてください:', FINAL_VIDEO_PATH)